In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier
import numpy as np
import tensorflow as tf

In [20]:
# Load dataset
data = load_iris()
X, y = data.data, data.target

In [21]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
import torch

# Check if MPS (Apple's GPU backend) is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [23]:
# Check if TensorFlow recognizes the GPU
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [24]:
def create_neural_network():
    model = Sequential([
        Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(8, activation='relu'),
        Dense(3, activation='softmax')  # 3 output classes for the Iris dataset
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
nn_model = KerasClassifier(model=create_neural_network, epochs=50, batch_size=8, verbose=0)

In [26]:
# Define other base models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('svc', SVC(probability=True, random_state=42)),
    ('nn', nn_model)
]

In [27]:
# Define meta-model
meta_model = LogisticRegression()

In [28]:
# Create stacking classifier with neural network as one of the base models
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)


In [29]:
# Train the stacking model
stacking_clf.fit(X_train, y_train)

/Users/nipunsingh/Documents/Project/Object Detection/detect/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2024-11-07 14:12:23.917320: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2024-11-07 14:12:23.917425: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2024-11-07 14:12:23.917436: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2024-11-07 14:12:23.917958: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-11-07 14:12:23.918540: I tensorflow/core/common_runtime/pluggable_device/pluggabl

StackingClassifier(cv=5,
                   estimators=[('rf',
                                RandomForestClassifier(n_estimators=10,
                                                       random_state=42)),
                               ('svc', SVC(probability=True, random_state=42)),
                               ('nn',
                                KerasClassifier(batch_size=8, epochs=50, model=<function create_neural_network at 0x3109e69e0>, verbose=0))],
                   final_estimator=LogisticRegression())

In [30]:
# Make predictions
y_pred = stacking_clf.predict(X_test)

In [31]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Stacking Classifier with Neural Network Accuracy: {accuracy:.2f}")

Stacking Classifier with Neural Network Accuracy: 1.00
